# schedule_raw_re_gift_customfields
Daily incremental sync for gift custom fields.
Run after `backfill_raw_re_gift_customfields.ipynb` has completed all 24 categories.

Uses `last_modified` watermark from `renxt_pipeline_state`.
Fetches all 24 categories with the date filter — small daily volume so finishes well within 24 hours.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
ENDPOINT_NAME  = "gift_customfields"
OUTPUT_DATASET = "raw_re_gift_customfields"   # replace with UUID if available
MERGE_KEY      = "id"

CATEGORIES = [
    'RG Acquisition Appeal',
    'RG Annual Receipt Number',
    'RG Billable',
    'RG Cancellation Method',
    'RG Cancellation Reason',
    'RG Payment Attempt',
    'RG Rejection Reason',
    'RG Save Attempt',
    'RG SignUp Age',
    'RG Signup Amount',
    'RG SignUp Date',
    'RG SignUp Location',
    'RG SignUp Name',
    'RG Signup Supplier',
    'RG SignUp Type',
    'RG Survey',
    'RG Survey Commitment',
    'RG Survey Longevity Confidence',
    'RG Upgrade Date',
    'RG Upgrade Last',
    'RG Upgrade Previous',
    'RG Upgrade Previous 2 Previous',
    'RG Verification Call',
    'Gift Acquisition Source',
]

GIFT_CF_URL = f"{API_BASE}/gift/v1/gifts/customfields"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ── Read watermark ────────────────────────────────────────────────────────────
state_df   = _read_pipeline_state()
since_date = _get_since_date(ENDPOINT_NAME, state_df)

print(f"Fetching gift custom fields modified since: {since_date}")
print(f"Categories to process: {len(CATEGORIES)}")
print()


In [ ]:
# ── Fetch all categories with date filter, upsert incrementally ───────────────
token_mgr    = TokenManager(interactive=False)
sess         = requests.Session()
total_rows   = 0
failed_cats  = []

for i, category in enumerate(CATEGORIES, 1):
    print(f"[{i}/{len(CATEGORIES)}] {category}")
    url      = GIFT_CF_URL
    params   = {"category": category, "include_inactive": "true",
                 "last_modified": since_date}
    cat_rows = 0
    page     = 0

    while True:
        page += 1
        resp = api_request_with_auth("GET", url, token_mgr=token_mgr,
                                     params=params, session=sess)
        _raise_for_status_with_body(resp,
            context=f"[gift_customfields/{category}] page={page}")

        payload          = resp.json() if resp.text else {}
        items, next_link = extract_items_and_next(payload)

        if items:
            df_page = pd.json_normalize(items, sep=".")
            df_page["category"]      = category
            df_page["pulled_at_utc"] = datetime.now(timezone.utc).isoformat()
            df_page["_endpoint"]     = "gift_customfields"
            df_page                  = domo_safe_cast(df_page)

            domo.write_dataframe(
                df_page,
                dataset=OUTPUT_DATASET,
                update_method="upsert",
                update_key=MERGE_KEY,
            )
            cat_rows  += len(df_page)
            total_rows += len(df_page)

        params = None   # next_link encodes params
        if not next_link:
            break
        url = next_link

    print(f"  ✅ {cat_rows:,} rows upserted")

print()
print(f"Total rows upserted: {total_rows:,}")


In [ ]:
# ── Update watermark ──────────────────────────────────────────────────────────
# Only advances if we got here without exception — all categories succeeded
_write_pipeline_state(ENDPOINT_NAME, "success", total_rows, state_df)
print(f"✅ Watermark updated for [{ENDPOINT_NAME}]")
